# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 05 — Random Forest

---

### Purpose
Train and evaluate a Random Forest classifier on all four feature sets.
Random Forest is expected to excel on dense stylometric and embedding features
where non-linear decision boundaries are beneficial.

### Objectives
1. Load all feature matrices
2. Train Random Forest on each feature set
3. Evaluate with full metrics suite
4. Generate confusion matrices, classification reports, ROC curves
5. Analyse Gini-based feature importance (especially for stylometric features)
6. Save trained models
7. Show prediction examples

### Notebook Outline
1. Imports
2. Configuration
3. Load TF-IDF Features
4. Load Character N-Gram Features
5. Load Stylometric Features
6. Load Embedding Features
7. Training
8. Evaluation
9. Confusion Matrix
10. Classification Report
11. ROC Curves
12. Gini Feature Importance
13. SHAP Analysis (Stylometric)
14. Model Saving
15. Prediction Examples
16. Notebook Summary

---

## 1. Imports

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import plotly.express as px

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.classifiers import RandomForestClassifier
from src.feature_engineering.utils import load_feature_matrix, setup_logger
from src.evaluation.evaluator import ModelEvaluator
from src.visualization.plots import (
    plot_confusion_matrix, plot_roc_curves, plot_feature_importance
)
from src.utils.helpers import (
    set_global_seed, train_test_val_split, load_yaml, make_output_dirs,
    display_metrics_table, print_section_header,
)

print('✅ All libraries imported successfully.')

---

## 2. Configuration

In [ ]:
cfg = load_yaml(PROJECT_ROOT / 'configs' / 'training.yaml')

RANDOM_SEED = cfg['random_seed']
TEST_SIZE   = cfg['evaluation']['test_size']
VAL_SIZE    = cfg['evaluation']['val_size']
RF_CFG      = cfg['random_forest']
FEAT_CFG    = cfg['features']

set_global_seed(RANDOM_SEED)

DIR_MODELS  = PROJECT_ROOT / cfg['output']['models_dir']
DIR_FIGURES = PROJECT_ROOT / cfg['output']['figures_dir']
DIR_OUTPUTS = PROJECT_ROOT / cfg['output']['outputs_dir']
make_output_dirs(DIR_MODELS, DIR_FIGURES, DIR_OUTPUTS)

setup_logger(str(PROJECT_ROOT / cfg['logging']['log_file']), cfg['logging']['level'])
logger = logging.getLogger(__name__)

print(f'Random Forest Config : {RF_CFG}')

---

## 3–6. Load Feature Matrices

In [ ]:
# ── Load all feature sets ──────────────────────────────────────────────────────
def load_and_split(matrix_path_str, labels_path_str, dense=False):
    """Load a feature matrix and split into train/val/test."""
    X, y = load_feature_matrix(
        PROJECT_ROOT / matrix_path_str,
        PROJECT_ROOT / labels_path_str,
    )
    if dense and sp.issparse(X):
        X = X.toarray()
    return train_test_val_split(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED)

# Load
X_tr_tf, X_val_tf, X_te_tf, y_tr_tf, y_val_tf, y_te_tf = load_and_split(
    FEAT_CFG['tfidf']['fingerprint'], FEAT_CFG['labels']['fingerprint']
)
X_tr_ch, X_val_ch, X_te_ch, y_tr_ch, y_val_ch, y_te_ch = load_and_split(
    FEAT_CFG['char']['fingerprint'], FEAT_CFG['labels']['fingerprint']
)
X_tr_st, X_val_st, X_te_st, y_tr_st, y_val_st, y_te_st = load_and_split(
    FEAT_CFG['style']['fingerprint'], FEAT_CFG['labels']['fingerprint'], dense=True
)
classes = np.load(
    str(PROJECT_ROOT / 'data' / 'features' / 'tfidf' / 'classes_tfidf_fingerprint.npy'),
    allow_pickle=True,
)

EMB_DIR = PROJECT_ROOT / 'data' / 'features' / 'embedding'
X_emb   = np.load(str(EMB_DIR / 'emb_fingerprint.npz'))['embeddings']
y_emb   = np.load(str(EMB_DIR / 'labels_emb_fingerprint.npy'))
X_tr_em, X_val_em, X_te_em, y_tr_em, y_val_em, y_te_em = train_test_val_split(
    X_emb, y_emb, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)

print('All feature matrices loaded.')
print(f'TF-IDF: {X_tr_tf.shape}  |  Char: {X_tr_ch.shape}  |  Style: {X_tr_st.shape}  |  Emb: {X_tr_em.shape}')

---

## 7. Training

In [ ]:
def run_random_forest(X_train, X_test, y_train, y_test, feature_set: str):
    """Train Random Forest and return (model, evaluator)."""
    model = RandomForestClassifier(cfg=RF_CFG)
    model.fit(X_train, y_train, feature_set=feature_set)

    evaluator = ModelEvaluator('random_forest', feature_set, classes)
    evaluator.evaluate(
        estimator=model.model,
        X_test=X_test,
        y_test=y_test,
        train_time=model.train_time_,
    )
    return model, evaluator

In [ ]:
print_section_header('Training Random Forest — TF-IDF Features')
rf_tfidf, eval_rf_tfidf = run_random_forest(X_tr_tf, X_te_tf, y_tr_tf, y_te_tf, 'tfidf')

print_section_header('Training Random Forest — Char N-Gram Features')
rf_char, eval_rf_char = run_random_forest(X_tr_ch, X_te_ch, y_tr_ch, y_te_ch, 'char')

print_section_header('Training Random Forest — Stylometric Features')
rf_style, eval_rf_style = run_random_forest(X_tr_st, X_te_st, y_tr_st, y_te_st, 'style')

print_section_header('Training Random Forest — Embedding Features')
rf_emb, eval_rf_emb = run_random_forest(X_tr_em, X_te_em, y_tr_em, y_te_em, 'embedding')

print('\n✅ All Random Forest variants trained.')

---

## 8. Evaluation

In [ ]:
results_rf = pd.DataFrame([
    eval_rf_tfidf.to_series(),
    eval_rf_char.to_series(),
    eval_rf_style.to_series(),
    eval_rf_emb.to_series(),
])

display_cols = ['model_name','feature_set','accuracy','precision_macro',
                'recall_macro','f1_macro','f1_weighted','roc_auc_macro',
                'train_time_s','pred_time_s','peak_memory_mb']
results_rf[display_cols].style.highlight_max(
    subset=['accuracy','f1_macro'], color='lightgreen'
).format(precision=4)

---

## 9. Confusion Matrix

In [ ]:
for evaluator, label in [
    (eval_rf_tfidf, 'TF-IDF'),
    (eval_rf_char,  'Char N-Gram'),
    (eval_rf_style, 'Stylometric'),
    (eval_rf_emb,   'Embedding'),
]:
    cm = np.array(evaluator.results_['confusion_matrix'])
    out_path = DIR_FIGURES / f'rf_cm_{evaluator.feature_set}.png'
    plot_confusion_matrix(
        cm=cm, class_names=list(classes),
        title=f'Random Forest — {label} — Confusion Matrix',
        out_path=out_path, normalize=True,
    )
    print(f'✅ {out_path.name}')

---

## 10. Classification Report

In [ ]:
best_rf_eval = max(
    [eval_rf_tfidf, eval_rf_char, eval_rf_style, eval_rf_emb],
    key=lambda e: e.results_['f1_macro'],
)
print(f'Best RF feature set: {best_rf_eval.feature_set}')
print(best_rf_eval.results_['classification_report'])
best_rf_eval.save_classification_report(
    DIR_OUTPUTS / f'rf_{best_rf_eval.feature_set}_classification_report.txt'
)

---

## 11. ROC Curves

In [ ]:
if best_rf_eval.results_.get('y_proba') is not None:
    roc_path = DIR_FIGURES / f'rf_roc_{best_rf_eval.feature_set}.png'
    plot_roc_curves(
        y_test=best_rf_eval.results_['y_test'],
        y_proba=best_rf_eval.results_['y_proba'],
        class_names=list(classes),
        title=f'Random Forest — {best_rf_eval.feature_set} — ROC Curves',
        out_path=roc_path,
    )
    print(f'✅ {roc_path.name}')

---

## 12. Gini Feature Importance

In [ ]:
# ── Gini importance for stylometric features (most interpretable) ──────────────
importances = rf_style.model.feature_importances_

from src.feature_engineering.stylometric_extractor import StylometricExtractor
from src.feature_engineering.utils import FeatureEngineeringConfig
from src.utils.helpers import load_yaml

feat_cfg = load_yaml(PROJECT_ROOT / 'configs' / 'feature_engineering.yaml')
style_ext = StylometricExtractor(cfg=feat_cfg.get('stylometric', {}))
style_feature_names = style_ext._build_feature_names()

importance_path = DIR_FIGURES / 'rf_style_feature_importance.png'
plot_feature_importance(
    importances=importances,
    feature_names=style_feature_names,
    title='Random Forest — Stylometric Feature Importance (Gini)',
    out_path=importance_path,
    top_n=20,
)
print(f'✅ Feature importance chart: {importance_path.name}')

---

## 13. SHAP Analysis (Stylometric)

In [ ]:
# ── SHAP TreeExplainer for Random Forest (stylometric features) ────────────────
# SHAP provides more rigorous feature attributions than Gini importance.
# Computes exact Shapley values for tree ensembles in polynomial time.

try:
    import shap

    # Use a subsample for speed (100 samples for explanation)
    n_shap_samples = min(100, X_te_st.shape[0])
    X_shap = X_te_st[:n_shap_samples]

    explainer = shap.TreeExplainer(rf_style.model)
    shap_values = explainer.shap_values(X_shap)

    print(f'SHAP values computed for {n_shap_samples} samples.')
    print(f'SHAP values shape: {len(shap_values)} classes × {X_shap.shape[0]} samples × {X_shap.shape[1]} features')

    # Summary plot for class 0
    plt.figure()
    shap.summary_plot(
        shap_values[0], X_shap,
        feature_names=style_feature_names,
        show=False,
    )
    shap_path = DIR_FIGURES / 'rf_shap_summary_stylometric.png'
    plt.savefig(shap_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'✅ SHAP summary plot saved: {shap_path.name}')

except ImportError:
    print('SHAP not installed. Run: pip install shap')

---

## 14. Model Saving

In [ ]:
for model, suffix in [
    (rf_tfidf, 'tfidf'),
    (rf_char,  'char'),
    (rf_style, 'style'),
    (rf_emb,   'embedding'),
]:
    saved_path = model.save(DIR_MODELS / 'random_forest', suffix=suffix)
    print(f'✅ Saved: {saved_path.name}')

---

## 15. Prediction Examples

In [ ]:
best_rf = {  # map feature_set → (model, X_test, y_test)
    'tfidf':     (rf_tfidf, X_te_tf, y_te_tf),
    'char':      (rf_char,  X_te_ch, y_te_ch),
    'style':     (rf_style, X_te_st, y_te_st),
    'embedding': (rf_emb,   X_te_em, y_te_em),
}[best_rf_eval.feature_set]

model, X_te, y_te = best_rf
y_pred_s  = model.predict(X_te[:10])
y_proba_s = model.predict_proba(X_te[:10])

pd.DataFrame({
    'True Label':      [classes[i] for i in y_te[:10]],
    'Predicted Label': [classes[i] for i in y_pred_s],
    'Correct':         y_te[:10] == y_pred_s,
    'Max Confidence':  np.max(y_proba_s, axis=1).round(4),
})

---

## 16. Notebook Summary

### Random Forest Results Summary

| Feature Set | Macro F1 | Weighted F1 | Train Time (s) | Memory (MB) |
|---|---|---|---|---|
| TF-IDF | *(populate)* | *(populate)* | *(populate)* | *(populate)* |
| Char N-Gram | *(populate)* | *(populate)* | *(populate)* | *(populate)* |
| Stylometric | *(populate)* | *(populate)* | *(populate)* | *(populate)* |
| Embeddings | *(populate)* | *(populate)* | *(populate)* | *(populate)* |

### Key Observations
- SHAP values reveal which stylometric features drive classification decisions
- Gini importance + SHAP provide complementary interpretability perspectives
- Expected best: Dense feature sets (stylometric, embeddings)

→ **Notebook 06**: XGBoost

---
*Fingerprint Project — Random Forest — Complete*